In [1]:
import pandas as pd
import numpy as np
df_raw = pd.read_excel('corporate_links_raw.xlsx', engine='openpyxl', header=None)

data = df_raw[0].str.split(',', expand=True)

df = data.iloc[1:].copy()
df.columns = data.iloc[0].tolist()

## Первоначальный Датасет:

In [2]:
df_not_normalized = df_raw

data = df_raw[0].str.split(',', expand=True)
df_not_normalized = data.iloc[1:].copy()
df_not_normalized.columns = data.iloc[0].tolist()

df_not_normalized

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date
1,Иванов И.И.,"ООО ""Ромашка""",7701234567,50%,Москва,Kommersant,12.03.2021
2,Иванов Иван Иванович,ООО Ромашка,7701234567,0.5,Москва,Kommersant,2021-03-12
3,Сидоров С.С,"ООО ""Ромашка""",7701234567,50 %,Москва,,15/03/2021
4,Петров Петр Петрович,"ООО ""Лютик""",7801234567,100%,Санкт-Петербург,RBC,01-04-2020
5,Петров П.П.,"ООО ""Лютик""",7801234567,30%,СПб,Forbes,2022/05/10
6,Иванов И.И.,"ООО ""Лютик""",7801234567,30 %,Москва,Forbes,10.05.2022
7,Смирнов А.А.,"АО ""Вектор""",5401234567,25%,Новосибирск,Kommersant,2019-07-01
8,Смирнов Алексей Алексеевич,АО Вектор,5401234567,0.25,Новосибирск,,01.07.2019
9,Кузнецова Е.В.,"ООО ""Альфа""",,40%,Москва,RBC,05.06.2020
10,Орлов Д.Д.,"ООО ""Альфа""",7723456789,70%,Москва,Kommersant,2020-06-05


## 1.1 
### Привести ФИО к формату: Фамилия И.О.

In [3]:
# функция по преобразованию фамилий в необходимый вид Фамилия И.О.

def normalize_fio(fio):
    if fio == '' or pd.isna(fio):
        return np.nan

    fio = ' '.join(str(fio).split())
    splitted = fio.split()
    num_parts = len(splitted)

    if fio.count('.') >= 2:
        if num_parts >= 3:
            return f"{splitted[0]} {splitted[1]}{splitted[2]}"
        else:
            return fio

    if num_parts == 3:
        return f"{splitted[0]} {splitted[1][0]}.{splitted[2][0]}."
    elif num_parts == 2:
        return f"{splitted[0]} {splitted[1][0]}."
    elif num_parts == 1:
        return splitted[0]
    else:
        return fio

In [4]:
# "аплаю" функцией и перезаписываю колонку 

df['FIO_owner'] = df['FIO_owner'].apply(normalize_fio)

## 1.2 
### Привести названия компаний к единому виду (без кавычек, без лишних пробелов)

In [5]:
# функция по преобразованию имен компаний

def new_company_name(company_name):
    if company_name == '' or pd.isna(company_name):
        return np.nan
    new_comp_name = ''
    parts_to_delete = '"' 

    company_name = company_name.strip()

    for i in range(len(company_name)):
        if company_name[i] not in parts_to_delete:
            new_comp_name += company_name[i]

    return new_comp_name


df['Company_name'] = df['Company_name'].apply(new_company_name)

## 1.3

### Привести ИНН к строковому типу
### Найти записи с отсутствующим ИНН
### Обозначить их отдельно

In [6]:
# функция по преобразованию ИНН 

def inn_to_string(inn):
    if inn == '':
        return np.nan
    inn = str(inn).strip()
    return inn
    

df['INN_company'] = df['INN_company'].apply(inn_to_string)

In [7]:
# смотрю пропуски по ИНН и вывожу их через print 

missing_inn = df['INN_company'].isna() | (df['INN_company'] == '') | (df['INN_company'] == ' ')
missing_inn_res = df[missing_inn].copy()

print(f"Количество записей без ИНН: {len(missing_inn_res)}")

Количество записей без ИНН: 1


## 1.4
### Привести Ownership к единому формату: в процентах (float) 0.5 → 50%

In [8]:
# функция по преобразованию Ownership

def to_correct_owners(owner):
    if owner == '' or pd.isna(owner):
        return None

    if '.' in owner:
        int_owner = float(owner) * 100
        return int_owner
    else:
        owner = owner.replace('%', '')
        return float(owner)

df['Ownership'] = df['Ownership'].apply(to_correct_owners)

## 1.5 
### убрать пробелы
### пустые значения выделить

In [9]:
df['Region'] = df['Region'].str.replace(' ', '')

idx = df[df['Region'] == 'СПб'].index
df.loc[idx, 'Region'] = 'Санкт-Петербург'

# добавляю наллы 
df['Region'] = df['Region'].replace('', np.nan)

In [10]:
df['Source'] = df['Source'].str.replace(' ', '') 

df['Source'] = df['Source'].replace('', np.nan)


## 1.6
### Привести даты к единому формату: YYYY-MM-DD

In [11]:
# функция по преобразованию дат 

def to_clean_date(date):
    if pd.isna(date):
        return np.nan
    
    date = str(date).strip()
    
    if '.' in date:
        splitted = date.split('.')
    elif '/' in date:
        splitted = date.split('/')
    elif '-' in date:
        splitted = date.split('-')
        
    # тут сплитованный массив с датами некст нужно запушить год в конец и привести к единому виду

    if len(splitted[0]) == 4:
        year = splitted[0]

        month = splitted[1]

        day = splitted[2]

        new_date = day + '.' + month + '.' + year 

        return new_date

    else:
        return '.'.join(splitted)
            
df['Ownership_date'] = df['Ownership_date'].apply(to_clean_date)

In [12]:
df['Ownership_date'] = pd.to_datetime(df['Ownership_date'], dayfirst=True)

## Итоговый Датасет после преобразований: 

In [13]:
df

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date
1,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12
2,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12
3,Сидоров С.,ООО Ромашка,7701234567,50.0,Москва,NaN,2021-03-15
4,Петров П.П.,ООО Лютик,7801234567,100.0,Санкт-Петербург,RBC,2020-04-01
5,Петров П.П.,ООО Лютик,7801234567,30.0,Санкт-Петербург,Forbes,2022-05-10
6,Иванов И.И.,ООО Лютик,7801234567,30.0,Москва,Forbes,2022-05-10
7,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,Kommersant,2019-07-01
8,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,NaN,2019-07-01
9,Кузнецова Е.В.,ООО Альфа,NaN,40.0,Москва,RBC,2020-06-05
10,Орлов Д.Д.,ООО Альфа,7723456789,70.0,Москва,Kommersant,2020-06-05


## 2.1 Анализ данных
### Найти компании, где суммарная доля владельцев >100%


In [14]:
df

,FIO_owner,Company_name,INN_company,Ownership,Region,Source,Ownership_date
1,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12
2,Иванов И.И.,ООО Ромашка,7701234567,50.0,Москва,Kommersant,2021-03-12
3,Сидоров С.,ООО Ромашка,7701234567,50.0,Москва,NaN,2021-03-15
4,Петров П.П.,ООО Лютик,7801234567,100.0,Санкт-Петербург,RBC,2020-04-01
5,Петров П.П.,ООО Лютик,7801234567,30.0,Санкт-Петербург,Forbes,2022-05-10
6,Иванов И.И.,ООО Лютик,7801234567,30.0,Москва,Forbes,2022-05-10
7,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,Kommersant,2019-07-01
8,Смирнов А.А.,АО Вектор,5401234567,25.0,Новосибирск,NaN,2019-07-01
9,Кузнецова Е.В.,ООО Альфа,NaN,40.0,Москва,RBC,2020-06-05
10,Орлов Д.Д.,ООО Альфа,7723456789,70.0,Москва,Kommersant,2020-06-05


In [15]:
# избавлюсь от дубликата (2 строка)

df = df.drop_duplicates()

In [16]:
# удалю 8 строку потому что это тоже дубликат

df = df.drop(8)

In [17]:
df_companies = df.groupby(['Company_name'])['Ownership'].sum().reset_index()

df_companies_filtered = df_companies.query('Ownership > 100')

In [18]:
# компании, где суммарная доля владельцев >100%

df_companies_filtered

,Company_name,Ownership
1,ООО Альфа,145.0
3,ООО Лютик,160.0


## 2.2 
### Найти компании, где доли менялись во времени

In [19]:
# фильтрую где компании и владельцы с нескольими записями и где значения по долям менялись 

changing_owners = df.groupby(['Company_name', 'FIO_owner']).filter(
    lambda x: (len(x) > 1) and (x['Ownership'].nunique() > 1)
)

changing_owners = changing_owners.sort_values(['Company_name', 'FIO_owner', 'Ownership_date'])

changing_owners[['Company_name', 'FIO_owner', 'Ownership_date', 'Ownership']]

,Company_name,FIO_owner,Ownership_date,Ownership
10,ООО Альфа,Орлов Д.Д.,2020-06-05,70.0
11,ООО Альфа,Орлов Д.Д.,2021-08-01,35.0
4,ООО Лютик,Петров П.П.,2020-04-01,100.0
5,ООО Лютик,Петров П.П.,2022-05-10,30.0


## 2.3
### Найти владельцев, участвующих более чем в одной компании

In [20]:
owners_companies = df.groupby('FIO_owner')['Company_name'].nunique().reset_index()
owners_companies.columns = ['FIO_owner', 'companies_count']

multi_company_owners = owners_companies[owners_companies['companies_count'] > 1]

In [21]:
# владельцы, участвующие более чем в одной компании

multi_company_owners

,FIO_owner,companies_count
0,Иванов И.И.,3


## 3

### _Какие данные вызывают сомнение?_

1) Логические противоречия в долях владения
   
2) В компании «ООО Лютик» сумма долей Петрова П.П. (100% + 30%) превышает 100%. Это невозможно для одного момента времени, хотя, возможно, отражает изменение во времени. Требуется проверка корректности учёта дат.

3) В компании «ООО Альфа» у Орлова Д.Д. указаны доли 70% и 35% – также суммарно больше 100% без указания на изменение во времени (если даты не учитывать).

4) В компании «ООО Альфа» у Кузнецовой Е.В. нет ИНН, хотя компания зарегистрирована и у других владельцев ИНН есть. Это может быть ошибкой ввода или неполнотой данных источника.

5) ФИО: «Сидоров С.С» (без последней точки), «Орлов Д. Д.» (лишний пробел). Проценты: «50%», «50 %», «0.5», «50.0». Даты: 12.03.2021, 2021-03-12, 15/03/2021 – создают путаницу день/месяц. Регионы: «СПб», «Санкт-Петербург» – несогласованность.

### _Где возможны ошибки источников?_

1) Отсутствие источника (NaN)
Записи без указания источника: строки 3 (Сидоров), 8 (Смирнов), 13 (Иванов в ООО Гамма).
   
2) Kommersant содержит две записи по Смирнову А.А. в компании «АО Вектор» с одинаковой датой и долей, но одна из них не имеет Source (строка 8). 

3) Неполнота данных в рамках источника
Kommersant: строка 12 (Лебедев) содержит пропуск в поле Ownership (доля владения). Источник предоставил неполную информацию.
RBC: строка 9 (Кузнецова) не содержит ИНН, хотя компания указана. Это может быть ошибкой источника или пропуском при сборе.

4) Технические ошибки, связанные с форматом данных
Хотя в представленной таблице данные уже нормализованы, в исходных файлах могли быть проблемы, характерные для конкретных источников (например, разные разделители дат, лишние пробелы, запись чисел с плавающей точкой), что косвенно указывает на небрежность при подготовке данных.